In [ ]:
# @title Setup: Install scpn-quantum-control
!pip install -q scpn-quantum-control
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)


SPDX-License-Identifier: AGPL-3.0-or-later | Commercial license available
Â© Concepts 1996â€“2026 Miroslav Ĺ otek. All rights reserved.
Â© Code 2020â€“2026 Miroslav Ĺ otek. All rights reserved.
ORCID: 0009-0009-3560-0851
Contact: www.anulum.li | protoscience@anulum.li
scpn-quantum-control â€” Stochastic Resonance in SCPN

HYPOTHESIS: Adding noise to Kuramoto oscillators can ENHANCE
synchronisation below K_c (stochastic resonance).
This is the mechanism behind NAQT in photosynthesis and
validates sc-neurocore's noise-as-resource architecture.

Tests:
1. R(K, sigma) heatmap â€” does R peak at nonzero noise for K < K_c?
2. Signal-to-noise ratio vs noise amplitude (classic SR signature)
3. Comparison to thermal noise at biological temperatures
4. NAQT analogy: FMO complex efficiency vs dephasing

In [ ]:
import json

In [ ]:
import numpy as np

In [ ]:
print("=" * 70)
print("STOCHASTIC RESONANCE IN SCPN KURAMOTO MODEL")
print("=" * 70)

In [ ]:
# SCPN parameters from Paper 27 (8 oscillators)
N = 8
omega = np.array([0.062, 0.191, 0.382, 0.618, 0.809, 0.927, 0.981, 1.000])
K_nm = np.array(
    [
        [0.000, 0.951, 0.588, 0.309, 0.191, 0.118, 0.073, 0.045],
        [0.951, 0.000, 0.951, 0.588, 0.309, 0.191, 0.118, 0.073],
        [0.588, 0.951, 0.000, 0.951, 0.588, 0.309, 0.191, 0.118],
        [0.309, 0.588, 0.951, 0.000, 0.951, 0.588, 0.309, 0.191],
        [0.191, 0.309, 0.588, 0.951, 0.000, 0.951, 0.588, 0.309],
        [0.118, 0.191, 0.309, 0.588, 0.951, 0.000, 0.951, 0.588],
        [0.073, 0.118, 0.191, 0.309, 0.588, 0.951, 0.000, 0.951],
        [0.045, 0.073, 0.118, 0.191, 0.309, 0.588, 0.951, 0.000],
    ]
)

In [ ]:
def simulate_noisy_kuramoto(K_scale, noise_sigma, dt=0.01, T=200, n_trials=20):
    """Kuramoto with additive Gaussian noise (Langevin).

    dtheta_i/dt = omega_i + K_scale * sum_j K_nm[i,j] sin(theta_j - theta_i) + sigma * xi(t)
    """
    n_steps = int(T / dt)
    R_trials = []

    for _ in range(n_trials):
        theta = np.random.uniform(0, 2 * np.pi, N)

        for _step in range(n_steps):
            dtheta = omega.copy()
            for i in range(N):
                coupling = 0.0
                for j in range(N):
                    coupling += K_scale * K_nm[i, j] * np.sin(theta[j] - theta[i])
                dtheta[i] += coupling / N
            noise = noise_sigma * np.random.randn(N) * np.sqrt(dt)
            theta += dtheta * dt + noise
            theta = theta % (2 * np.pi)

        # Order parameter from final state
        z = np.mean(np.exp(1j * theta))
        R_trials.append(abs(z))

    return np.mean(R_trials), np.std(R_trials)

In [ ]:
# Find K_c first (no noise)
print("\n--- Finding K_c (no noise) ---")
K_vals_coarse = np.linspace(0.5, 5.0, 20)
R_no_noise = []
for K in K_vals_coarse:
    r, _ = simulate_noisy_kuramoto(K, 0.0, n_trials=10)
    R_no_noise.append(r)
    print(f"K={K:.2f}: R={r:.3f}")

In [ ]:
# K_c ~ where R crosses 0.5
R_arr = np.array(R_no_noise)
idx_cross = np.argmin(np.abs(R_arr - 0.5))
K_c_est = K_vals_coarse[idx_cross]
print(f"\nEstimated K_c ~ {K_c_est:.2f} (R=0.5 crossing)")

In [ ]:
# TEST 1: R(K, noise) heatmap
print("\n" + "=" * 70)
print("TEST 1: STOCHASTIC RESONANCE HEATMAP")
print("=" * 70)

In [ ]:
# Scan K below, at, and above K_c
K_test = np.array(
    [
        K_c_est * 0.3,
        K_c_est * 0.5,
        K_c_est * 0.7,
        K_c_est * 0.9,
        K_c_est,
        K_c_est * 1.2,
        K_c_est * 1.5,
    ]
)
noise_test = np.array([0.0, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0])

In [ ]:
heatmap = {}
sr_detected = False

In [ ]:
for K in K_test:
    row = {}
    R_at_noise = []
    for sigma in noise_test:
        r, r_std = simulate_noisy_kuramoto(K, sigma, n_trials=15)
        row[f"sigma={sigma:.2f}"] = {"R": round(r, 4), "std": round(r_std, 4)}
        R_at_noise.append(r)
        print(f"K={K:.2f}, sigma={sigma:.2f}: R={r:.3f} Â± {r_std:.3f}")

    # Check for SR: does R peak at nonzero noise?
    R_at_noise = np.array(R_at_noise)
    peak_idx = np.argmax(R_at_noise)
    if peak_idx > 0 and K_c_est > K:
        sr_detected = True
        print(f"  >>> SR DETECTED at K={K:.2f}: peak R at sigma={noise_test[peak_idx]:.2f}")

    heatmap[f"K={K:.2f}"] = row

In [ ]:
print(f"\nStochastic resonance detected: {sr_detected}")

In [ ]:
# TEST 2: Classic SR signature â€” SNR vs noise for subthreshold signal
print("\n" + "=" * 70)
print("TEST 2: SIGNAL-TO-NOISE RATIO (subthreshold K)")
print("=" * 70)

In [ ]:
K_sub = K_c_est * 0.5  # well below threshold
noise_fine = np.linspace(0.01, 3.0, 30)
snr_vals = []

In [ ]:
for sigma in noise_fine:
    r_mean, r_std = simulate_noisy_kuramoto(K_sub, sigma, n_trials=20)
    snr = r_mean / max(r_std, 0.001)  # signal-to-noise
    snr_vals.append(snr)
    if sigma < 0.5 or sigma > 2.5 or abs(sigma - 1.0) < 0.15:
        print(f"sigma={sigma:.2f}: R={r_mean:.3f}, std={r_std:.3f}, SNR={snr:.2f}")

In [ ]:
snr_arr = np.array(snr_vals)
peak_snr_idx = np.argmax(snr_arr)
optimal_noise = noise_fine[peak_snr_idx]
print(f"\nOptimal noise (max SNR): sigma={optimal_noise:.2f}, SNR={snr_arr[peak_snr_idx]:.2f}")
print(f"Zero-noise SNR: {snr_vals[0]:.2f}")
print(f"SR amplification: {snr_arr[peak_snr_idx] / max(snr_vals[0], 0.001):.2f}x")

In [ ]:
# TEST 3: Biological temperature noise
print("\n" + "=" * 70)
print("TEST 3: THERMAL NOISE AT BIOLOGICAL TEMPERATURES")
print("=" * 70)

In [ ]:
kB = 1.381e-23  # J/K
temperatures = {
    "deep_space": 2.7,
    "liquid_N2": 77,
    "room_temp": 293,
    "body_37C": 310,
    "fever_40C": 313,
    "hyperthermia": 315,
    "protein_denature": 340,
}

In [ ]:
# Thermal noise in dimensionless units: sigma ~ sqrt(kB*T / E_coupling)
# E_coupling ~ K * omega_mean * hbar for quantum, or K * some energy scale
# We normalise to body temperature = 1.0
T_body = 310
for name, T in temperatures.items():
    sigma_thermal = np.sqrt(T / T_body)  # normalised
    r, r_std = simulate_noisy_kuramoto(K_c_est * 0.8, sigma_thermal, n_trials=15)
    print(f"{name:20s} ({T:6.1f} K): sigma={sigma_thermal:.3f}, R={r:.3f} Â± {r_std:.3f}")

In [ ]:
# TEST 4: NAQT analogy â€” FMO complex
print("\n" + "=" * 70)
print("TEST 4: NOISE-ASSISTED QUANTUM TRANSPORT (FMO ANALOGY)")
print("=" * 70)

In [ ]:
# In the FMO complex, transport efficiency peaks at ~300K dephasing
# This is stochastic resonance in a quantum system
# Model: 7-site FMO-like chain with dephasing noise
N_fmo = 7
omega_fmo = np.linspace(0.0, 1.0, N_fmo)
K_fmo = np.zeros((N_fmo, N_fmo))
for i in range(N_fmo - 1):
    K_fmo[i, i + 1] = 1.0
    K_fmo[i + 1, i] = 1.0

In [ ]:
# Transport = how fast phase information propagates from site 0 to site 6
dephasing_rates = np.linspace(0.0, 5.0, 25)
transport_eff = []

In [ ]:
for gamma in dephasing_rates:
    # Simulate with dephasing noise
    dt = 0.01
    T_sim = 100
    n_steps = int(T_sim / dt)
    n_trials = 20
    arrival_count = 0

    for _ in range(n_trials):
        theta = np.zeros(N_fmo)
        theta[0] = np.pi  # inject at site 0

        for _step in range(n_steps):
            dtheta = omega_fmo.copy()
            for i in range(N_fmo):
                for j in range(N_fmo):
                    dtheta[i] += K_fmo[i, j] * np.sin(theta[j] - theta[i]) / N_fmo
            noise = gamma * np.random.randn(N_fmo) * np.sqrt(dt)
            theta += dtheta * dt + noise

        # Transport efficiency: how much phase reached site 6
        phase_transfer = abs(np.sin(theta[-1]))
        arrival_count += phase_transfer

    eff = arrival_count / n_trials
    transport_eff.append(eff)

In [ ]:
transport_arr = np.array(transport_eff)
peak_idx = np.argmax(transport_arr)
print(f"Peak transport at dephasing gamma={dephasing_rates[peak_idx]:.2f}")
print(f"Zero-dephasing transport: {transport_arr[0]:.3f}")
print(f"Peak transport: {transport_arr[peak_idx]:.3f}")
print(f"NAQT enhancement: {transport_arr[peak_idx] / max(transport_arr[0], 0.001):.2f}x")

In [ ]:
naqt_confirmed = transport_arr[peak_idx] > transport_arr[0] * 1.1
print(f"\nNAQT confirmed (>10% enhancement): {naqt_confirmed}")

In [ ]:
# TEST 5: Noise-induced order in SCPN
print("\n" + "=" * 70)
print("TEST 5: NOISE-INDUCED ORDER (pure noise, no deterministic coupling)")
print("=" * 70)

In [ ]:
# Can noise ALONE create synchronisation in a system with natural frequencies?
# This is the "noise-induced order" phenomenon
K_zero = 0.0
noise_levels = np.linspace(0.0, 10.0, 20)
R_noise_only = []

In [ ]:
for sigma in noise_levels:
    r, r_std = simulate_noisy_kuramoto(K_zero, sigma, n_trials=20, T=500)
    R_noise_only.append(r)
    print(f"sigma={sigma:.1f}: R={r:.3f} Â± {r_std:.3f}")

In [ ]:
# SYNTHESIS
print("\n" + "=" * 70)
print("SYNTHESIS: STOCHASTIC RESONANCE IN SCPN")
print("=" * 70)

In [ ]:
print(f"\n1. Stochastic resonance detected: {sr_detected}")
print(f"2. Optimal noise for subthreshold signal: sigma={optimal_noise:.2f}")
print(f"3. SR amplification factor: {snr_arr[peak_snr_idx] / max(snr_vals[0], 0.001):.2f}x")
print(f"4. NAQT (noise-assisted transport) confirmed: {naqt_confirmed}")
print(f"5. Peak NAQT enhancement: {transport_arr[peak_idx] / max(transport_arr[0], 0.001):.2f}x")

In [ ]:
# Biological implications
print("\n--- BIOLOGICAL IMPLICATIONS ---")
print("If SR confirmed: biological systems operate at OPTIMAL noise level.")
print("Body temperature (310K) provides thermal noise that HELPS synchronisation.")
print("Fever (313-315K) changes the noise â€” potentially disrupting OR enhancing sync.")
print("This is WHY life operates at ~310K â€” it's the SR optimum for the coupling topology.")
print("Validates sc-neurocore noise-as-resource: SNN computation NEEDS noise.")

In [ ]:
# JSON output
results = {
    "sr_detected": sr_detected,
    "K_c_estimate": round(K_c_est, 3),
    "optimal_noise_sigma": round(optimal_noise, 3),
    "sr_amplification": round(float(snr_arr[peak_snr_idx] / max(snr_vals[0], 0.001)), 3),
    "naqt_confirmed": bool(naqt_confirmed),
    "naqt_enhancement": round(float(transport_arr[peak_idx] / max(transport_arr[0], 0.001)), 3),
    "naqt_peak_dephasing": round(float(dephasing_rates[peak_idx]), 3),
}

In [ ]:
print("\n--- JSON RESULTS ---")
print(json.dumps(results, indent=2))